In [27]:
%pip install -U langchain-classic langchain-ollama langchain-core

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   -------------------- ------------------- 0.5/1.0 MB 699.0 kB/s eta 0:00:01
   ------------------------------ --------- 0.8/1.0 MB 819.2 kB/s eta 0:00:01
   ------------------------------ --------- 0.8/1.0 MB 819.2 kB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 817.8 kB/s  0:00:01

  Attempting uninstall: langchain-text-splitters

    Found existing installation: langchain-text-splitters 0.3.11

    Uninstalling langchain-text-splitters-0.3.11:

      Successfully uninstalled langchain-text-splitters-0.3.11

   ---------------------------------------- 0/2 [langchain-text-splitters]
   -------------------- ---------------

In [28]:
# Import standard libraries for file handling and text processing
import os, pathlib, textwrap, glob

# Load documents from various sources (URLs, text files, PDFs)
from langchain_community.document_loaders import UnstructuredURLLoader, TextLoader, PyPDFLoader


# Split long texts into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# FAISS vector database
from langchain_community.vectorstores import FAISS

# Embedding models
from langchain_community.embeddings import SentenceTransformerEmbeddings

# Local language model through Ollama
from langchain_ollama import OllamaLLM

# Prompt templates
from langchain_core.prompts import PromptTemplate

from langchain_classic.chains import ConversationalRetrievalChain


print("✅ Libraries imported - you're good to go!")

✅ Libraries imported - you're good to go!


In [26]:
from langchain_community.document_loaders import UnstructuredURLLoader, TextLoader
print("Other loader imports successful")

Other loader imports successful


In [23]:
%pip install -U langchain

  Using cached langchain_core-1.6.10-py3-none-any.whl.metadata (4.8 kB)
Using cached langchain_core-1.6.10-py3-none-any.whl (573 kB)

   ---- ----------------------------------- 1/9 [websockets]
   ---- ----------------------------------- 1/9 [websockets]
  Attempting uninstall: langchain-core
   ---- ----------------------------------- 1/9 [websockets]
    Found existing installation: langchain-core 1.6.9
   ---- ----------------------------------- 1/9 [websockets]
   ------------- -------------------------- 3/9 [langchain-core]
    Uninstalling langchain-core-1.6.9:
   ------------- -------------------------- 3/9 [langchain-core]
      Successfully uninstalled langchain-core-1.6.9
   ------------- -------------------------- 3/9 [langchain-core]
   ------------- -------------------------- 3/9 [langchain-core]
   ------------- -------------------------- 3/9 [langchain-core]
   ------------- -------------------------- 3/9 [langchain-core]
   ------------- -------------------------- 3/9 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.3.24 requires langchain<1.0.0,>=0.3.25, but you have langchain 1.4.5 which is incompatible.
langchain-community 0.3.24 requires langchain-core<1.0.0,>=0.3.59, but you have langchain-core 1.6.10 which is incompatible.


In [5]:
import glob

pdf_paths = glob.glob("data/*.pdf")
raw_docs = []

print(f"Found {len(pdf_paths)} PDF files.")

for i, path in enumerate(pdf_paths, start=1):
    print(f"[{i}/{len(pdf_paths)}] Loading: {path}", flush=True)

    try:
        pages = PyPDFLoader(path).load()
        raw_docs.extend(pages)
        print(f"    Loaded {len(pages)} pages.", flush=True)

    except Exception as e:
        print(f"    Failed: {e}", flush=True)




tmp = PyPDFLoader(pdf_paths[0]).load()
print(tmp[0].page_content)
for path in pdf_paths:
    tmp = PyPDFLoader(path).load()
    raw_docs.extend(tmp)

print(f"Loaded {len(raw_docs)} PDF pages from {len(pdf_paths)} files.")

raw_docs[0]


Found 38 PDF files.
[1/38] Loading: data\2023.emnlp-main.557.pdf
    Loaded 14 pages.
[2/38] Loading: data\A Review of Faithfulness Metrics for Hallucination Assessment in Large Language Models.pdf
    Loaded 13 pages.
[3/38] Loading: data\Adaptive-ensemble-attack-breaking-Large-Multimodal-Models-via-dynamic-caption-selection-and-weighted-gradients_2026_Springer-Science-and-Business-Media-Deutschland-GmbH.pdf
    Loaded 24 pages.
[4/38] Loading: data\Advancing-adversarial-and-LLM-robustness-in-trustworthy-AI-a-comprehensive-survey_2026_Springer-Nature.pdf
    Loaded 51 pages.
[5/38] Loading: data\Are-LLMs-Aware-that-Some-Questions-are-not-Openended_2024_Association-for-Computational-Linguistics-ACL.pdf
    Loaded 11 pages.
[6/38] Loading: data\black box consistency under same expression.pdf
    Loaded 12 pages.
[7/38] Loading: data\Can-Multimodal-LLMs-Perform-Time-Series-Anomaly-Detection_2026_Association-for-Computing-Machinery-Inc.pdf
    Loaded 12 pages.
[8/38] Loading: data\Compari

KeyboardInterrupt: 

In [17]:
%pip install -U langchain-huggingface sentence-transformers


Note: you may need to restart the kernel to use updated packages.


In [6]:
chunks = []
text_splitter = RecursiveCharacterTextSplitter(
  chunk_size=1000,
  chunk_overlap=200,
  separators=["\n\n", "\n", " ", ""],
)

chunks = text_splitter.split_documents(raw_docs)
print(f"{len(chunks)} chunk ready for embedding")

3822 chunk ready for embedding


In [7]:
embedding_vector = []

embeddings = SentenceTransformerEmbeddings(
      model_name="thenlper/gte-small",
      model_kwargs={"device": "cuda"},
      encode_kwargs={"batch_size": 64}
)

text = "Hello World"
embedding_vector = embeddings.embed_query(text)

print(len(embedding_vector))

C:\Users\aradh\AppData\Local\Temp\ipykernel_38684\57904823.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

384


In [8]:
import time

texts = [chunk.page_content for chunk in chunks[:100]]

start = time.time()
vectors = embeddings.embed_documents(texts)

print("Vectors generated:", len(vectors))
print("Time taken:", round(time.time() - start, 2), "seconds")

Vectors generated: 100
Time taken: 0.53 seconds


In [13]:
import torch
print("Embedding device:", embeddings.client.device)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Embedding device: cuda:0
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


In [9]:
print("Total PDF pages:", len(raw_docs))
print("Total chunks:", len(chunks))

print("\nSample chunk:")
print(chunks[0].page_content[500:1000])

Total PDF pages: 714
Total chunks: 3822

Sample chunk:
GPT-3 are capable of generating highly
fluent responses to a wide variety of user
prompts. However, LLMs are known to hal-
lucinate facts and make non-factual statements
which can undermine trust in their output. Ex-
isting fact-checking approaches either require
access to the output probability distribution
(which may not be available for systems such
as ChatGPT) or external databases that are in-
terfaced via separate, often complex, modules.
In this work, we propose "SelfCheckGPT", a


In [10]:

# Expected steps:
    # 1. Build the FAISS index from the list of document chunks and their embeddings.
    # 2. Create a retriever object with a suitable k value (e.g., 8).
    # 3. Save the vector store locally (e.g., under "faiss_index").
    # 4. Print a short confirmation showing how many embeddings were stored.
start = time.time()
vectordb = FAISS.from_documents(chunks, embeddings)
print("Time taken:", round(time.time() - start, 2), "seconds")
retriever = vectordb.as_retriever(search_type="similarity", search_kwargs = {"k":4})
vectordb.save_local("faiss_index")
print("✅ Vector store with", vectordb.index.ntotal, "embeddings")

Time taken: 10.44 seconds
✅ Vector store with 3822 embeddings


In [11]:
print("PDF pages:", len(raw_docs))
print("Text chunks:", len(chunks))
print("FAISS embeddings:", vectordb.index.ntotal)

PDF pages: 714
Text chunks: 3822
FAISS embeddings: 3822


In [12]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)


PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
CUDA version: 13.0


In [13]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")


PyTorch: 2.14.1+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
GPU memory: 7.96 GB


In [16]:

import sys
print(sys.executable)

c:\Users\aradh\miniconda3\envs\rag-chatbot\python.exe


In [19]:
%pip install -U langchain-ollama

  Using cached langchain_ollama-1.1.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached ollama-0.6.3-py3-none-any.whl.metadata (7.6 kB)
Using cached langchain_ollama-1.1.0-py3-none-any.whl (31 kB)
Using cached ollama-0.6.3-py3-none-any.whl (16 kB)

   ---------------------------------------- 2/2 [langchain-ollama]

Note: you may need to restart the kernel to use updated packages.


In [20]:
from langchain_ollama import OllamaLLM

print("Import successful!")

Import successful!


In [21]:
llm = OllamaLLM(model="gemma3:1b", temperature=0.1)

text = "what is 1+1?"
response = llm.invoke(text)
response

"1 + 1 = 2\n\nIt's a fundamental concept in arithmetic! 😊\n"

In [29]:

SYSTEM_TEMPLATE = """
You are a Research Assistant specializing in healthcare-domain
Retrieval-Augmented Generation (RAG) and hallucination detection.

Your task is to answer the user's questions using only the
information provided in the retrieved research-paper context.

RULES:

1. EVIDENCE-BASED ANSWERS
   Use only the retrieved context to answer questions.
   Do not introduce external knowledge, assumptions, or unsupported claims.

2. HANDLE MISSING INFORMATION
   If the context does not contain sufficient evidence to answer,
   respond: "I don't know based on the retrieved documents."
   Do not guess, infer unsupported facts, or fabricate information.

3. HEALTHCARE SAFETY
   Treat healthcare-related information as research evidence,
   not as personalized medical advice.
   Do not provide diagnoses, treatment recommendations, or
   clinical conclusions unless explicitly supported by the context.
   Clearly state when the retrieved evidence is insufficient.

4. RESEARCH-FOCUSED RESPONSES
   When relevant, distinguish between:
   - Research objectives
   - Methodology
   - Hallucination detection methods
   - Evaluation metrics
   - Experimental results and findings
   - Limitations and future work

   Do not assume that a method is effective merely because
   a paper proposes it. Report effectiveness only when supported
   by the retrieved evidence.

5. COMPARATIVE ANALYSIS
   When comparing methods, compare only the methods and findings
   supported by the context.
   Identify differences in evaluation datasets, metrics, and
   experimental settings when this information is available.
   Do not claim that one method is superior unless the evidence
   supports that conclusion.

6. CITATIONS AND TRACEABILITY
   Cite supporting sources using their metadata whenever available,
   preferably in the format [Source: paper title, page number].
   Never invent paper titles, page numbers, authors, or citations.
   If source metadata is unavailable, explicitly avoid fabricating it.

7. RESPONSE STYLE
   Be concise, precise, objective, and academically appropriate.
   Use simple explanations for technical concepts when needed.
   Prefer evidence from the retrieved context over general statements.
   Clearly distinguish reported findings from authors' interpretations.

RETRIEVED CONTEXT:
{context}

USER QUESTION:
{question}

ANSWER:
"""


In [30]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=SYSTEM_TEMPLATE
)
llm = OllamaLLM(
    model="gemma3:1b",
    temperature=0.1
)
chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    combine_docs_chain_kwargs={"prompt": prompt},
    return_source_documents=True
)

In [31]:

response = chain.invoke({
    "question": "What hallucination detection methods are discussed in the retrieved papers?",
    "chat_history": []
})

print("Answer:")
print(response["answer"])

print("\nSource documents:")
for doc in response["source_documents"]:
    print("Source:", doc.metadata)
    print(doc.page_content[:300])
    print("-" * 50)


Answer:
Xu et al. (2024) have proven that hallucinations are inevitable in LLMs. Existing studies on hallucination detection for LLMs can be broadly divided into two categories: retrieval-based methods (Min et al., 2023; Liu et al., 2023) and sampling-based methods (Mündler et al., 2023; Manakul et al., 2023). However, retrieval-based methods depend heavily on external knowledge, and sampling-based methods require multiple responses from the LLM, making them costly. To address these issues, Xu et al. (2024) propose a novel method.


Source documents:
Source: {'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-01-08T08:45:53+00:00', 'author': '', 'keywords': '', 'moddate': '2024-01-08T08:45:53+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\Enhancing-UncertaintyBased-Hallucination-Detection-with-Stronger-Focus_2023_Asso

In [32]:

# Research-focused test questions for healthcare RAG
test_questions = [
    "What hallucination detection methods are discussed in the retrieved research papers?",

    "How do the discussed hallucination detection methods differ in their evaluation metrics and experimental results?",

    "What limitations or challenges of hallucination detection in healthcare RAG systems are reported in the papers?",

    "What datasets are used to evaluate the hallucination detection methods, and what findings are reported?"
]

# Initialize chat history
chat_history = []

# Test the RAG chain
for question in test_questions:
    result = chain.invoke({
        "question": question,
        "chat_history": chat_history
    })

    print(f"Q: {question}")
    print(f"A: {result['answer']}\n")

    # Display source papers used to generate the answer
    print("Sources:")
    for doc in result.get("source_documents", []):
        print(doc.metadata)
    print("-" * 80)

    # Save this exchange for conversational follow-up
    chat_history.append((question, result["answer"]))


Q: What hallucination detection methods are discussed in the retrieved research papers?
A: Xu et al. (2024) have proven that hallucinations are inevitable in LLMs. Existing studies on hallucination detection for LLMs can be broadly divided into two categories: retrieval-based methods (Min et al., 2023; Liu et al., 2023) and sampling-based methods (Mündler et al., 2023; Manakul et al., 2023). However, retrieval-based methods depend heavily on external knowledge, and sampling-based methods require multiple responses from the LLM, making them costly. To address these issues, Xu et al. (2024) propose a novel method.


Sources:
{'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-01-08T08:45:53+00:00', 'author': '', 'keywords': '', 'moddate': '2024-01-08T08:45:53+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\Enhancing-Unce